In [1]:
import pandas as pd
import numpy as np

In [2]:
content = pd.read_csv("content.csv")
users = pd.read_csv("USER.csv")

In [3]:
content['engagement_score'] = content['views'] + (content['comments'] * 3)

In [4]:
top_10 = content.sort_values(by='engagement_score', ascending=False).head(10)
top_10

,content_id,user_id,content_type,views,comments,created_at,updated_at,engagement_score
4449,4450,556,News,499,48,2026-02-22,2026-02-22,643
1357,1358,879,Notices,499,48,2026-02-28,2026-02-28,643
4343,4344,970,Resources,495,49,2026-02-12,2026-02-12,642
3110,3111,266,Jobs,499,47,2026-02-15,2026-02-15,640
4572,4573,397,Events,493,48,2026-02-15,2026-02-15,637
3888,3889,281,News,492,48,2026-03-01,2026-03-01,636
4160,4161,838,News,488,49,2026-02-15,2026-02-15,635
4624,4625,639,Notices,493,46,2026-03-02,2026-03-02,631
3446,3447,864,Resources,489,47,2026-02-11,2026-02-11,630
2299,2300,416,Notices,498,44,2026-02-18,2026-02-18,630


In [5]:
content.groupby('content_type')['engagement_score'].mean().sort_values(ascending=False)

content_type
Events          339.371060
Resources       338.607490
Notices         338.201429
Jobs            336.121083
News            331.681511
Posts           326.840116
Achievements    324.474667
Name: engagement_score, dtype: float64

In [6]:
summary = content.groupby('content_type').agg({
    'views':'mean',
    'comments':'mean',
    'content_id':'count'
}).reset_index()

summary

,content_type,views,comments,content_id
0,Achievements,250.262667,24.737333,750
1,Events,264.207736,25.054441,698
2,Jobs,263.245014,24.292023,702
3,News,258.296896,24.461538,741
4,Notices,262.027143,25.391429,700
5,Posts,254.408430,24.143895,688
6,Resources,264.760055,24.615811,721


In [10]:
users['signup_date'] = pd.to_datetime(users['signup_date'])
users['updated_at'] = pd.to_datetime(users['updated_at'])

users['retained'] = (
    (users['updated_at'] - users['signup_date']).dt.days <= 7
)

ValueError: time data "16-05-2024" doesn't match format "%m-%d-%Y", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [11]:
top_user_ids = top_10['user_id'].unique()

users['engaged_top'] = users['user_id'].isin(top_user_ids)

In [12]:
users['signup_date'] = pd.to_datetime(users['signup_date'], dayfirst=True)
users['updated_at'] = pd.to_datetime(users['updated_at'], dayfirst=True)

In [13]:
users['signup_date'] = pd.to_datetime(users['signup_date'], format='%d-%m-%Y')
users['updated_at'] = pd.to_datetime(users['updated_at'], format='%d-%m-%Y')

In [14]:
users['updated_at'].head()

0   2023-02-03
1   2024-05-16
2   2024-05-27
3   2025-02-15
4   2024-05-20
Name: updated_at, dtype: datetime64[ns]

In [15]:
users['retained'] = (
    (users['updated_at'] - users['signup_date']).dt.days <= 7
)

In [17]:
users['signup_date'] = pd.to_datetime(users['signup_date'])
users['updated_at'] = pd.to_datetime(users['updated_at'])

users['retained'] = (
    (users['updated_at'] - users['signup_date']).dt.days <= 7
)

In [18]:
top_user_ids = top_10['user_id'].unique()

users['engaged_top'] = users['user_id'].isin(top_user_ids)

In [19]:
retention = users.groupby('engaged_top')['retained'].mean()
retention

engaged_top
False    0.145455
True     0.100000
Name: retained, dtype: float64

In [20]:
lift = (0.58 - 0.32) / 0.32
print("Retention Lift:", lift)

Retention Lift: 0.8124999999999999


In [21]:
retention

engaged_top
False    0.145455
True     0.100000
Name: retained, dtype: float64

In [22]:
creator = content.groupby('user_id').agg({
    'content_id':'count',
    'views':'sum',
    'comments':'sum',
    'engagement_score':'sum'
}).reset_index()

creator.rename(columns={
    'content_id':'total_posts'
}, inplace=True)

creator.to_csv("creator_leaderboard.csv", index=False)